<a href="https://colab.research.google.com/github/VasilisPapageorgiou/Amortization-of-Risk-Indicators/blob/main/Amortization_RemovalTimes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# REMOVAL HISTORY EXPERIMENT
# Model A: removal history only
# Model B: removal history + N/Nmax + i0/N
# Exact targets: infection count distribution + extinction moments
# Automatic resume/checkpoint every 20 minutes
# ============================================================

import os, time, pickle, random
import numpy as np
import pandas as pd

from functools import lru_cache
from scipy.sparse import coo_matrix, csc_matrix
from scipy.sparse.linalg import splu

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

# ------------------------------------------------------------
# Google Drive
# ------------------------------------------------------------

try:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = "/content/drive/MyDrive/SIRS_removal_MLP"
except:
    ROOT = "./SIRS_removal_MLP"

os.makedirs(ROOT, exist_ok=True)

# ------------------------------------------------------------
# Settings
# ------------------------------------------------------------

SEED = 1234

R_TRAIN = 10_000
R_VAL   = 1_000
R_TEST  = 1_000

N_GRID = np.arange(40, 401, 20)
NMAX = int(N_GRID.max())

BETA_RANGE  = (0.30, 1.50)
GAMMA_RANGE = (0.20, 1.00)
OMEGA_RANGE = (0.02, 0.50)

# Only removal times observed before T_OBS are supplied to the MLP
T_OBS = 5.0
BINS = 20

LAMBDA_RHO = 1.0
LAMBDA_TAU = 0.02

BATCH = 64
MAX_EPOCHS = 500
PATIENCE = 20
LR = 1e-3
WD = 1e-6

SAVE_SECONDS = 20 * 60

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ============================================================
# Ask whether to resume or start from the beginning
# ============================================================

def choose_run_mode():

    while True:

        print("\nChoose how to run the experiment:")
        print("1: Resume from saved progress")
        print("2: Start from the beginning")

        choice = input("Enter 1 or 2: ").strip()

        if choice == "1":
            print("\nRESUME selected.")
            return "resume"

        if choice == "2":
            print("\nSTART FROM BEGINNING selected.")
            return "start"

        print("Invalid choice. Please enter 1 or 2.")


RUN_MODE = choose_run_mode()

if RUN_MODE == "start":

    files_to_remove = [
        "train_R10000.pkl",
        "validation.pkl",
        "test.pkl",
        "removal_only_checkpoint.pt",
        "removal_only_best.pt",
        "removal_plus_N_i0_checkpoint.pt",
        "removal_plus_N_i0_best.pt",
        "test_results.csv",
        "summary.csv"
    ]

    removed = 0

    for filename in files_to_remove:

        path = os.path.join(ROOT, filename)

        if os.path.exists(path):
            os.remove(path)
            removed += 1

        tmp_path = path + ".tmp"

        if os.path.exists(tmp_path):
            os.remove(tmp_path)

    print(f"Removed {removed} saved experiment files.")
    print("The experiment will start from the beginning.\n")

else:

    print("Existing datasets and checkpoints will be used when available.\n")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ============================================================
# Utilities
# ============================================================

def atomic_pickle(obj, path):
    tmp = path + ".tmp"
    with open(tmp, "wb") as f:
        pickle.dump(obj, f, protocol=pickle.HIGHEST_PROTOCOL)
    os.replace(tmp, path)

def atomic_torch(obj, path):
    tmp = path + ".tmp"
    torch.save(obj, tmp)
    os.replace(tmp, path)

# ============================================================
# SIRS state structure
# ============================================================

@lru_cache(None)
def topology(N):

    states = [
        (s, i)
        for i in range(1, N + 1)
        for s in range(N - i + 1)
    ]

    idx = {x: k for k, x in enumerate(states)}

    svec = np.array([x[0] for x in states], dtype=float)
    ivec = np.array([x[1] for x in states], dtype=float)
    rvec = N - svec - ivec

    inf_r, inf_c, inf_coef = [], [], []
    rec_r, rec_c, rec_coef = [], [], []
    wan_r, wan_c, wan_coef = [], [], []

    for k, (s, i) in enumerate(states):

        if s > 0:
            inf_r.append(k)
            inf_c.append(idx[(s - 1, i + 1)])
            inf_coef.append(s * i / N)

        if i > 1:
            rec_r.append(k)
            rec_c.append(idx[(s, i - 1)])
            rec_coef.append(i)

        rr = N - s - i
        if rr > 0:
            wan_r.append(k)
            wan_c.append(idx[(s + 1, i)])
            wan_coef.append(rr)

    return (
        states, idx,
        svec, ivec, rvec,
        np.asarray(inf_r), np.asarray(inf_c), np.asarray(inf_coef),
        np.asarray(rec_r), np.asarray(rec_c), np.asarray(rec_coef),
        np.asarray(wan_r), np.asarray(wan_c), np.asarray(wan_coef)
    )

# ============================================================
# Exact Markovian targets
# ============================================================

def exact_targets(beta, gamma, omega, N, i0):

    (
        states, idx,
        svec, ivec, rvec,
        ir, ic, ia,
        rr, rc, ra,
        wr, wc, wa
    ) = topology(N)

    M = len(states)

    diag = -(
        beta * svec * ivec / N
        + gamma * ivec
        + omega * rvec
    )

    d0_rows = list(range(M))
    d0_cols = list(range(M))
    d0_data = list(diag)

    d0_rows += list(rr)
    d0_cols += list(rc)
    d0_data += list(gamma * ra)

    d0_rows += list(wr)
    d0_cols += list(wc)
    d0_data += list(omega * wa)

    D0 = csc_matrix(
        (d0_data, (d0_rows, d0_cols)),
        shape=(M, M)
    )

    D1 = csc_matrix(
        (beta * ia, (ir, ic)),
        shape=(M, M)
    )

    T = D0 + D1

    q = np.zeros(M)
    q[ivec == 1] = gamma

    x0 = idx[(N - i0, i0)]

    # Infection count distribution
    lu0 = splu(-D0)

    v = lu0.solve(q)

    p = np.zeros(N + 2)
    p[0] = v[x0]

    for k in range(1, N + 1):
        v = lu0.solve(D1 @ v)
        p[k] = v[x0]

    p[N + 1] = max(0.0, 1.0 - p[:N + 1].sum())

    total = p.sum()
    if total > 0:
        p /= total

    # Extinction time mean and variance
    luT = splu(-T)

    one = np.ones(M)

    m1 = luT.solve(one)
    m2 = 2.0 * luT.solve(m1)

    mean_tau = m1[x0]
    var_tau = max(0.0, m2[x0] - mean_tau ** 2)

    return p, mean_tau, var_tau

# ============================================================
# Partial epidemic trajectory
# Only removal times up to T_OBS are retained
# ============================================================

def simulate_removals(beta, gamma, omega, N, i0, rng):

    s = N - i0
    i = i0
    t = 0.0

    removals = []

    while i > 0 and t < T_OBS:

        r = N - s - i

        rates = np.array([
            beta * s * i / N,
            gamma * i,
            omega * r
        ])

        total = rates.sum()

        if total <= 0:
            break

        dt = rng.exponential(1.0 / total)

        if t + dt > T_OBS:
            break

        t += dt

        u = rng.uniform() * total

        if u < rates[0]:

            if s > 0:
                s -= 1
                i += 1

        elif u < rates[0] + rates[1]:

            i -= 1
            removals.append(t)

        else:

            if r > 0:
                s += 1

    # Fixed dimensional representation
    edges = np.linspace(0.0, T_OBS, BINS + 1)

    counts, _ = np.histogram(removals, bins=edges)

    # Global scaling only, so Model A is not explicitly supplied N
    hist = counts.astype(np.float32) / NMAX

    return hist

# ============================================================
# Configuration sampling
# ============================================================

def sample_configuration(rng):

    N = int(rng.choice(N_GRID))

    beta  = rng.uniform(*BETA_RANGE)
    gamma = rng.uniform(*GAMMA_RANGE)
    omega = rng.uniform(*OMEGA_RANGE)

    if rng.uniform() < 0.25:
        i0 = 1
    else:
        frac = rng.uniform(0.02, 0.20)
        i0 = int(np.clip(round(frac * N), 1, N))

    return beta, gamma, omega, N, i0

# ============================================================
# Generate dataset with resume
# ============================================================

def build_split(name, target_size, seed_base):

    path = os.path.join(ROOT, f"{name}.pkl")

    if RUN_MODE == "resume" and os.path.exists(path):
        with open(path, "rb") as f:
            records = pickle.load(f)
        print(f"{name}: resumed at {len(records)}/{target_size}")
    else:
        records = []
        print(f"{name}: starting from 0/{target_size}")

    last_save = time.time()

    while len(records) < target_size:

        k = len(records)
        attempt = 0

        while True:

            rng = np.random.default_rng(seed_base + 1000 * k + attempt)

            beta, gamma, omega, N, i0 = sample_configuration(rng)

            try:

                p, mtau, vtau = exact_targets(
                    beta, gamma, omega, N, i0
                )

                hist = simulate_removals(
                    beta, gamma, omega, N, i0, rng
                )

                if (
                    np.all(np.isfinite(p))
                    and np.isfinite(mtau)
                    and np.isfinite(vtau)
                    and mtau > 0
                    and vtau >= 0
                ):
                    break

            except Exception:
                pass

            attempt += 1

        records.append({
            "theta": np.array([beta, gamma, omega], dtype=np.float32),
            "N": N,
            "i0": i0,
            "hist": hist,
            "p": p.astype(np.float32),
            "mean_tau": float(mtau),
            "var_tau": float(vtau)
        })

        if time.time() - last_save >= SAVE_SECONDS:
            atomic_pickle(records, path)
            print(f"{name}: saved {len(records)}/{target_size}")
            last_save = time.time()

    atomic_pickle(records, path)

    print(f"{name}: complete")

    return records


train_records = build_split("train_R10000", R_TRAIN, 10_000_000)
val_records   = build_split("validation",   R_VAL,   20_000_000)
test_records  = build_split("test",         R_TEST,  30_000_000)

# ============================================================
# Tensor conversion
# ============================================================

def pack(records):

    n = len(records)

    H = np.stack([x["hist"] for x in records])

    N = np.array([x["N"] for x in records])
    i0 = np.array([x["i0"] for x in records])

    P = np.zeros((n, NMAX + 2), dtype=np.float32)

    for r, x in enumerate(records):
        P[r, :len(x["p"])] = x["p"]

    Ytau = np.stack([
        [
            np.log1p(x["mean_tau"]),
            np.log1p(x["var_tau"])
        ]
        for x in records
    ]).astype(np.float32)

    return TensorDataset(
        torch.tensor(H),
        torch.tensor(N, dtype=torch.long),
        torch.tensor(i0, dtype=torch.long),
        torch.tensor(P),
        torch.tensor(Ytau)
    )


TRAIN = pack(train_records)
VAL   = pack(val_records)
TEST  = pack(test_records)

# ============================================================
# Networks
# ============================================================

class MLP(nn.Module):

    def __init__(self, din, dout, output):

        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(din, 128),
            nn.SiLU(),
            nn.Linear(128, 128),
            nn.SiLU(),
            nn.Linear(128, 128),
            nn.SiLU(),
            nn.Linear(128, dout)
        )

        self.output = output

    def forward(self, x):

        z = self.net(x)

        if self.output == "sigmoid":
            return torch.sigmoid(z)

        if self.output == "softplus":
            return nn.functional.softplus(z)

        return z

# ============================================================
# Features
# ============================================================

def base_features(hist, N, i0, include_size):

    if not include_size:
        return hist

    return torch.cat([
        hist,
        (N.float() / NMAX).unsqueeze(1),
        (i0.float() / N.float()).unsqueeze(1)
    ], dim=1)

# ============================================================
# Hazard reconstruction
# ============================================================

def predict_distribution(net, hist, N, i0, include_size):

    B = hist.shape[0]
    L = NMAX + 1

    base = base_features(hist, N, i0, include_size)

    c = (
        torch.arange(L, device=hist.device).float()
        / NMAX
    )

    X = base[:, None, :].expand(-1, L, -1)

    C = c[None, :, None].expand(B, -1, -1)

    X = torch.cat([X, C], dim=2).reshape(B * L, -1)

    h = net(X).reshape(B, L)

    q = 1.0 - h

    surv_before = torch.cat([
        torch.ones(B, 1, device=hist.device),
        torch.cumprod(q[:, :-1], dim=1)
    ], dim=1)

    masses = h * surv_before

    surv_after = torch.cumprod(q, dim=1)

    phat = torch.zeros(
        B, NMAX + 2,
        device=hist.device
    )

    phat[:, :L] = masses

    rows = torch.arange(B, device=hist.device)

    phat[rows, N + 1] = surv_after[rows, N]

    cols = torch.arange(NMAX + 2, device=hist.device)

    mask = cols[None, :] <= (N[:, None] + 1)

    phat = phat * mask

    return phat, mask

# ============================================================
# Loss
# ============================================================

def loss_components(
    hazard_net,
    tau_net,
    batch,
    include_size
):

    hist, N, i0, p, ytau = [
        x.to(DEVICE) for x in batch
    ]

    phat, mask = predict_distribution(
        hazard_net, hist, N, i0, include_size
    )

    # PMF error
    Lpmf = (
        ((phat - p) ** 2 * mask)
        .sum(dim=1)
        .mean()
    )

    # Tail risk error
    p0 = p * mask
    ph0 = phat * mask

    tail = (
        torch.flip(
            torch.cumsum(torch.flip(p0, [1]), dim=1),
            [1]
        )
        - p0
    )

    tailhat = (
        torch.flip(
            torch.cumsum(torch.flip(ph0, [1]), dim=1),
            [1]
        )
        - ph0
    )

    ccols = torch.arange(
        NMAX + 1,
        device=DEVICE
    )

    tailmask = ccols[None, :] <= N[:, None]

    Lrho = (
        (
            (tailhat[:, :NMAX + 1] - tail[:, :NMAX + 1]) ** 2
            * tailmask
        )
        .sum(dim=1)
        / (N.float() + 1.0)
    ).mean()

    # Extinction time
    Xtau = base_features(
        hist, N, i0, include_size
    )

    yhat = tau_net(Xtau)

    W = 1.0 / (1.0 + ytau ** 2)

    Ltau = (
        ((yhat - ytau) ** 2 * W)
        .sum(dim=1)
        .mean()
    )

    total = (
        Lpmf
        + LAMBDA_RHO * Lrho
        + LAMBDA_TAU * Ltau
    )

    return total, Lpmf, Lrho, Ltau

# ============================================================
# Training with automatic resume
# ============================================================

def train_model(name, include_size):

    hazard_dim = BINS + 1 + (2 if include_size else 0)
    tau_dim    = BINS     + (2 if include_size else 0)

    hazard = MLP(
        hazard_dim, 1, "sigmoid"
    ).to(DEVICE)

    tau = MLP(
        tau_dim, 2, "softplus"
    ).to(DEVICE)

    optimizer = torch.optim.AdamW(
        list(hazard.parameters()) + list(tau.parameters()),
        lr=LR,
        weight_decay=WD
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        factor=0.5,
        patience=15
    )

    ckpt_path = os.path.join(ROOT, f"{name}_checkpoint.pt")
    best_path = os.path.join(ROOT, f"{name}_best.pt")

    start_epoch = 0
    best_val = np.inf
    bad_epochs = 0

    if RUN_MODE == "resume" and os.path.exists(ckpt_path):

        ck = torch.load(
            ckpt_path,
            map_location=DEVICE
        )

        hazard.load_state_dict(ck["hazard"])
        tau.load_state_dict(ck["tau"])
        optimizer.load_state_dict(ck["optimizer"])
        scheduler.load_state_dict(ck["scheduler"])

        start_epoch = ck["epoch"]
        best_val = ck["best_val"]
        bad_epochs = ck["bad_epochs"]

        print(f"{name}: resumed at epoch {start_epoch}")

    else:

        print(f"{name}: starting training from epoch 0")

    train_loader = DataLoader(
        TRAIN,
        batch_size=BATCH,
        shuffle=True
    )

    val_loader = DataLoader(
        VAL,
        batch_size=BATCH,
        shuffle=False
    )

    last_save = time.time()

    for epoch in range(start_epoch, MAX_EPOCHS):

        hazard.train()
        tau.train()

        for batch in train_loader:

            optimizer.zero_grad()

            L, _, _, _ = loss_components(
                hazard, tau, batch, include_size
            )

            L.backward()

            torch.nn.utils.clip_grad_norm_(
                list(hazard.parameters()) + list(tau.parameters()),
                5.0
            )

            optimizer.step()

            if time.time() - last_save >= SAVE_SECONDS:

                atomic_torch({
                    "epoch": epoch,
                    "hazard": hazard.state_dict(),
                    "tau": tau.state_dict(),
                    "optimizer": optimizer.state_dict(),
                    "scheduler": scheduler.state_dict(),
                    "best_val": best_val,
                    "bad_epochs": bad_epochs
                }, ckpt_path)

                print(
                    f"{name}: 20 minute checkpoint "
                    f"at epoch {epoch + 1}"
                )

                last_save = time.time()

        # Validation
        hazard.eval()
        tau.eval()

        vals = []

        with torch.no_grad():

            for batch in val_loader:

                L, _, _, _ = loss_components(
                    hazard, tau, batch, include_size
                )

                vals.append(L.item())

        val_loss = np.mean(vals)

        scheduler.step(val_loss)

        if val_loss < best_val - 1e-6:

            best_val = val_loss
            bad_epochs = 0

            atomic_torch({
                "hazard": hazard.state_dict(),
                "tau": tau.state_dict()
            }, best_path)

        else:
            bad_epochs += 1

        # Save every epoch as well
        atomic_torch({
            "epoch": epoch + 1,
            "hazard": hazard.state_dict(),
            "tau": tau.state_dict(),
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict(),
            "best_val": best_val,
            "bad_epochs": bad_epochs
        }, ckpt_path)

        print(
            f"{name:24s} "
            f"epoch={epoch + 1:3d} "
            f"val={val_loss:.6f}"
        )

        if bad_epochs >= PATIENCE:
            break

    best = torch.load(
        best_path,
        map_location=DEVICE
    )

    hazard.load_state_dict(best["hazard"])
    tau.load_state_dict(best["tau"])

    return hazard, tau

# ============================================================
# Fit both experiments
# ============================================================

hazard_A, tau_A = train_model(
    "removal_only",
    include_size=False
)

hazard_B, tau_B = train_model(
    "removal_plus_N_i0",
    include_size=True
)

# ============================================================
# Test evaluation
# ============================================================

def evaluate(name, hazard, tau, include_size):

    loader = DataLoader(
        TEST,
        batch_size=BATCH,
        shuffle=False
    )

    rows = []

    hazard.eval()
    tau.eval()

    with torch.no_grad():

        for batch in loader:

            hist, N, i0, p, ytau = [
                x.to(DEVICE) for x in batch
            ]

            phat, mask = predict_distribution(
                hazard, hist, N, i0, include_size
            )

            Xtau = base_features(
                hist, N, i0, include_size
            )

            yhat = tau(Xtau)

            for j in range(len(N)):

                n = int(N[j])

                pt = p[j, :n + 2].cpu().numpy()
                pp = phat[j, :n + 2].cpu().numpy()

                E2 = np.linalg.norm(pp - pt)

                rt = np.flip(
                    np.cumsum(np.flip(pt))
                ) - pt

                rp = np.flip(
                    np.cumsum(np.flip(pp))
                ) - pp

                Erho = np.max(
                    np.abs(rp[:n + 1] - rt[:n + 1])
                )

                Eover = abs(
                    pp[n + 1] - pt[n + 1]
                )

                nz = pt > 0

                DKL = np.sum(
                    pt[nz]
                    * np.log(
                        pt[nz]
                        / np.clip(pp[nz], 1e-12, None)
                    )
                )

                true_tau = np.expm1(
                    ytau[j].cpu().numpy()
                )

                pred_tau = np.expm1(
                    yhat[j].cpu().numpy()
                )

                rows.append({
                    "model": name,
                    "N": n,
                    "E2": E2,
                    "Erho": Erho,
                    "Eover": Eover,
                    "DKL": DKL,
                    "RelMeanTau":
                        abs(pred_tau[0] - true_tau[0])
                        / max(true_tau[0], 1e-12),
                    "RelVarTau":
                        abs(pred_tau[1] - true_tau[1])
                        / max(true_tau[1], 1e-12)
                })

    return pd.DataFrame(rows)


res_A = evaluate(
    "Removal only",
    hazard_A,
    tau_A,
    False
)

res_B = evaluate(
    "Removal + N + i0",
    hazard_B,
    tau_B,
    True
)

results = pd.concat(
    [res_A, res_B],
    ignore_index=True
)

results.to_csv(
    os.path.join(ROOT, "test_results.csv"),
    index=False
)

# ============================================================
# Final comparison
# ============================================================

metrics = [
    "E2",
    "Erho",
    "Eover",
    "DKL",
    "RelMeanTau",
    "RelVarTau"
]

summary = (
    results
    .groupby("model")[metrics]
    .agg([
        "median",
        lambda x: x.quantile(0.25),
        lambda x: x.quantile(0.75),
        lambda x: x.quantile(0.95)
    ])
)

print("\nFINAL TEST COMPARISON\n")
print(summary)

summary.to_csv(
    os.path.join(ROOT, "summary.csv")
)